# 🟡 Solution: Convex Polygon IoU

**Algorithm:** Sutherland-Hodgman clip + shoelace area

**Reduction:** `iou = inter_area / (area_a + area_b - inter_area)` where the intersection polygon comes from clipping `poly_a` against `poly_b`.

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import numpy as np

In [ ]:
# algorithm: Sutherland-Hodgman clip + shoelace (self-contained)

import numpy as np

def convex_polygon_iou(poly_a, poly_b):
    def _area(v):
        v=np.asarray(v,dtype=float); x,y=v[:,0],v[:,1]
        return 0.5*abs(np.sum(x*np.roll(y,-1)-np.roll(x,-1)*y))

    def _clip(subj, clip):
        subj=[np.array(v,dtype=float) for v in subj]
        clip=[np.array(v,dtype=float) for v in clip]
        def inside(p,c1,c2): return (c2[0]-c1[0])*(p[1]-c1[1])-(c2[1]-c1[1])*(p[0]-c1[0])>=0
        def inter(s,e,c1,c2):
            ds=e-s; dc=c2-c1; d=dc[0]*ds[1]-dc[1]*ds[0]
            if abs(d)<1e-12: return s.copy()
            return s+((s[0]-c1[0])*dc[1]-(s[1]-c1[1])*dc[0])/d*ds
        out=list(subj)
        for i in range(len(clip)):
            if not out: break
            c1,c2=clip[i],clip[(i+1)%len(clip)]; inp,out=out,[]
            for j in range(len(inp)):
                s,e=inp[j-1],inp[j]
                if inside(e,c1,c2):
                    if not inside(s,c1,c2): out.append(inter(s,e,c1,c2))
                    out.append(e.copy())
                elif inside(s,c1,c2): out.append(inter(s,e,c1,c2))
        return np.array(out,dtype=float) if out else np.zeros((0,2))

    poly_a=np.asarray(poly_a,dtype=float); poly_b=np.asarray(poly_b,dtype=float)
    inter=_clip(poly_a,poly_b)
    if len(inter)<3: return 0.0
    inter_area=_area(inter); area_a=_area(poly_a); area_b=_area(poly_b)
    union=area_a+area_b-inter_area
    return float(inter_area/union) if union>1e-12 else 0.0

In [ ]:
# 🔍 Verify solution
# Identical squares → IoU = 1.0
sq = np.array([[0.,0.],[2.,0.],[2.,2.],[0.,2.]])
print("Identical:", convex_polygon_iou(sq, sq.copy()))   # expect 1.0

# Non-overlapping → 0.0
far = np.array([[5.,5.],[6.,5.],[6.,6.],[5.,6.]])
print("No overlap:", convex_polygon_iou(sq, far))         # expect 0.0

# Half-overlapping → 1/3
b = np.array([[1.,0.],[3.,0.],[3.,2.],[1.,2.]])
print("Half overlap:", convex_polygon_iou(sq, b))         # expect 0.333...

In [ ]:
# ✅ Inline test suite
import numpy as np, time

# ── Test 1: identical → 1.0 ────────────────────────────────────────────────
sq = np.array([[0.,0.],[2.,0.],[2.,2.],[0.,2.]])
r1 = convex_polygon_iou(sq, sq.copy())
assert abs(r1 - 1.0) < 1e-6, f"Identical: {r1}"
print("Test 1 passed: identical polygons → IoU=1")

# ── Test 2: non-overlapping → 0.0 ─────────────────────────────────────────
far = np.array([[5.,5.],[6.,5.],[6.,6.],[5.,6.]])
r2 = convex_polygon_iou(sq, far)
assert abs(r2 - 0.0) < 1e-9, f"No overlap: {r2}"
print("Test 2 passed: non-overlapping → IoU=0")

# ── Test 3: half-overlapping → 1/3 ────────────────────────────────────────
b = np.array([[1.,0.],[3.,0.],[3.,2.],[1.,2.]])
r3 = convex_polygon_iou(sq, b)
assert abs(r3 - 1/3) < 1e-6, f"Half overlap: {r3:.6f} expected {1/3:.6f}"
print("Test 3 passed: half-overlap → IoU=1/3")

# ── Test 4: symmetry ───────────────────────────────────────────────────────
a4 = np.array([[0.,0.],[3.,0.],[3.,1.],[0.,1.]])
b4 = np.array([[1.,-1.],[4.,-1.],[4.,2.],[1.,2.]])
r4a = convex_polygon_iou(a4, b4); r4b = convex_polygon_iou(b4, a4)
assert abs(r4a - r4b) < 1e-9 and 0 <= r4a <= 1, f"Symmetry: {r4a} vs {r4b}"
print("Test 4 passed: IoU is symmetric")

# ── Test 5: 100 random pairs in [0,1] ─────────────────────────────────────
rng=np.random.default_rng(13)
def rand_sq(rng):
    x,y=rng.uniform(0,5),rng.uniform(0,5); s=rng.uniform(0.5,2)
    return np.array([[x,y],[x+s,y],[x+s,y+s],[x,y+s]])
t0=time.time()
for _ in range(100):
    r=convex_polygon_iou(rand_sq(rng), rand_sq(rng))
    assert 0<=r<=1+1e-6, f"IoU out of range: {r}"
elapsed=time.time()-t0
assert elapsed<3.0, f"Too slow: {elapsed:.2f}s"
print(f"Test 5 passed: 100 random pairs ({elapsed:.3f}s)")

print("\nAll tests passed!")